In [1]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 9db35d83-e86f-43e2-a041-3d59308fdeb3
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 9db35d83-e86f-43e2-a041-3d59308fdeb3 to get into ready status...
Session 9db35d83-e86f-43e2-a041-3d59308fdeb3 

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("GlueJob").getOrCreate()
from pyspark.sql.functions import *
from pyspark.sql.types import *


Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 8241d936-49b7-48a1-9dcf-c47ca028fd42
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 8241d936-49b7-48a1-9dcf-c47ca028fd42 to get into ready status...
Session 8241d936-49b7-48a1-9dcf-c47ca028fd42 has been created.



In [2]:
claims_df   = spark.read.parquet("s3://healthcareinsurance-bucket/bronze/claims/")
customer_df = spark.read.parquet("s3://healthcareinsurance-bucket/bronze/customer/")
hospital_df = spark.read.parquet("s3://healthcareinsurance-bucket/bronze/hospital/")
payments_df = spark.read.parquet("s3://healthcareinsurance-bucket/bronze/payments/")
policy_df   = spark.read.parquet("s3://healthcareinsurance-bucket/bronze/policy/")

In [11]:
def transformation_pipeline():

    c = customer_df.alias("c")
    p = policy_df.alias("p")
    cl = claims_df.alias("cl")
    h = hospital_df.alias("h")
    pay = payments_df.alias("pay")

    df = cl.join(p, col("cl.policy_id") == col("p.policy_id"), "inner") \
           .join(c, col("p.customer_id") == col("c.customer_id"), "inner") \
           .join(h, col("cl.hospital_id") == col("h.hospital_id"), "inner") \
           .join(pay, col("cl.claim_id") == col("pay.claim_id"), "inner")
  

    final_df = df.select(
        col("c.customer_id"),
        col("p.policy_id"),
        col("cl.claim_id"),
        col("c.firstname"),
        col("c.lastname"),
        col("c.city").alias("customer_city"),
        col("h.city").alias("hospital_city"),
        col("c.age"),
        col("c.gender"),
        col("pay.paid_amount"),
        col("cl.claim_amount"),
        col("p.policy_type"),
        col("cl.status"),
        col("h.hospital_name"),
        col("cl.claim_date"),
        col("p.start_date"),
        col("p.end_date"),
    )

    final_df = final_df.withColumn("claim_gap", col("claim_amount") - col("paid_amount")) \
        .withColumn("high_claim_flag", when(col("claim_amount") > 10000, 1).otherwise(0)) \
        .withColumn("approve_flag", when(lower(col("status")) == "approved", 1).otherwise(0)) \
        .withColumn("age_group",
                    when(col("age") < 30, "Young")
                    .when((col("age") >= 30) & (col("age") < 50), "Adult")
                    .otherwise("Old")) \
        .withColumn("claim_delay", datediff(col("claim_date"), col("start_date")))\
        .withColumn("ingestion_time",from_utc_timestamp(current_timestamp(),"Asia/Kolkata"))

    return final_df

In [4]:
def save_clean_data_in_parquet_format_in_silver(df):
    df.write.format("parquet").option('header',True).mode('overwrite').save(f"s3://healthcareinsurance-bucket/silver/final_df/")
    print("data saved to silver bucket folder")

In [6]:
final_df =transformation_pipeline()
save_clean_data_in_parquet_format_in_silver(final_df)

data saved to silver bucket folder


+-----------+---------+--------+---------+--------+-------------+-------------+---+------+-----------+------------+--------------+--------+-------------+----------+----------+----------+---------+---------------+------------+---------+-----------+--------------------+
|customer_id|policy_id|claim_id|firstname|lastname|customer_city|hospital_city|age|gender|paid_amount|claim_amount|   policy_type|  status|hospital_name|claim_date|start_date|  end_date|claim_gap|high_claim_flag|approve_flag|age_group|claim_delay|      ingestion_time|
+-----------+---------+--------+---------+--------+-------------+-------------+---+------+-----------+------------+--------------+--------+-------------+----------+----------+----------+---------+---------------+------------+---------+-----------+--------------------+
|       1947|    15012|   20002|   Anjali|   Singh|         Pune|    Bangalore| 63|     F|      87557|       26155|Family Floater|Rejected| Apollo_11451|21-04-2022|09-10-2021|09-10-2022|   -614